In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import scipy.io

# Create Model

In [2]:
class NavierStokes(nn.Module):
    def __init__(self):
        super(NavierStokes, self).__init__()

        #self.layers = layer
        self.net = self.initial_model()

        self.lambda1 = nn.Parameter(torch.tensor([0.5]))
        self.lambda2 = nn.Parameter(torch.tensor([0.5]))

        self.register_parameter("lambda1", self.lambda1)
        self.register_parameter("lambda2", self.lambda2)

    def initial_model(self):
        net = nn.Sequential(
            nn.Linear(3, 20),
            nn.Tanh(),
            nn.Linear(20, 20),
            nn.Tanh(),
            nn.Linear(20, 20),
            nn.Tanh(),
            nn.Linear(20, 20),
            nn.Tanh(),
            nn.Linear(20, 20),
            nn.Tanh(),
            nn.Linear(20, 20),
            nn.Tanh(),
            nn.Linear(20, 20),
            nn.Tanh(),
            nn.Linear(20, 20),
            nn.Tanh(),
            nn.Linear(20, 2)
        )
        return net
    
    def forward(self, x):
        return self.net(x)


# Define Grad and Error

In [3]:
def gradients(outputs, inputs):
    return torch.autograd.grad(outputs, inputs, grad_outputs=torch.ones_like(outputs), create_graph=True)

In [28]:
def pde(x_input_ts, y_input_ts, net):
    psi_and_p = net(x_input_ts)
    u_pre = gradients(psi_and_p[:,0:1], x_input_ts)[0][:,1:2]
    v_pre = -gradients(psi_and_p[:,0:1], x_input_ts)[0][:,0:1]

    u_pre_d = gradients(u_pre, x_input_ts)[0]
    u_t = u_pre_d[:, 2:3]
    u_x = u_pre_d[:, 0:1]
    u_y = u_pre_d[:, 1:2]
    u_xx = gradients(u_x, x_input_ts)[0][:,0:1]
    u_yy = gradients(u_y, x_input_ts)[0][:,1:2]

    v_pre_d = gradients(v_pre, x_input_ts)[0]
    v_t = v_pre_d[:, 2:3]
    v_x = v_pre_d[:, 0:1]
    v_y = v_pre_d[:, 1:2]
    v_xx = gradients(v_x, x_input_ts)[0][:,0:1]
    v_yy = gradients(v_y, x_input_ts)[0][:,1:2]

    p_pre = psi_and_p[:, 1:2]
    p_d = gradients(p_pre, x_input_ts)[0]
    p_x = p_d[:, 0:1]
    p_y = p_d[:, 1:2]

    f_u = u_t + net.lambda1[0]*(u_pre * u_x + v_pre *u_y) + p_x - net.lambda2[0] * (u_xx + u_yy)

    f_v = v_t + net.lambda1[0]*(u_pre * v_x + v_pre * v_y) + p_y - net.lambda2[0] * (v_xx + v_yy)

    return f_u, f_v, u_pre, v_pre

# Data Preparation

In [22]:
data = scipy.io.loadmat("cylinder_nektar_wake.mat")

U_star = data["U_star"]
P_star = data["p_star"]
X_star = data["X_star"]
t_star = data["t"]

N = X_star.shape[0]
T = t_star.shape[0]

XX = np.tile(X_star[:, 0:1], (1, T))
YY = np.tile(X_star[:, 1:2], (1, T))
TT = np.tile(t_star, (1, N))

UU = U_star[:, 0, :]
VV = U_star[:, 1, :]
PP = P_star

x = XX.flatten()[:, None]
y = YY.flatten()[:, None]
t = TT.flatten()[:, None]

u = UU.flatten()[:, None]
v = VV.flatten()[:, None]

idx = np.random.choice(N*T, 2000, replace=False)
x_train = x[idx, :]
y_train = y[idx, :]
t_train = t[idx, :]
u_train = u[idx, :]
v_train = v[idx, :]

x_input = np.concatenate([x_train, y_train, t_train], axis=1)
y_input = np.concatenate([u_train, v_train], axis=1)

# Create Model

In [33]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

x_input_ts = torch.tensor(x_input, dtype=torch.float32, requires_grad=True).to(device)
y_input_ts = torch.tensor(y_input, dtype=torch.float32, requires_grad=True).to(device)

model = NavierStokes().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 10000
for epoch in range(epochs):
    optimizer.zero_grad()
    
    f_u, f_v, u_pre, v_pre = pde(x_input_ts, y_input_ts, model)
    loss_fu = criterion(f_u, torch.zeros_like(f_u))
    loss_fv = criterion(f_v, torch.zeros_like(f_v))
    loss_u = criterion(u_pre, y_input_ts[:, 0:1])
    loss_v = criterion(v_pre, y_input_ts[:, 1:2])

    loss = loss_fu + loss_fv + loss_u + loss_v
    loss.backward()
    optimizer.step()

    if (epoch+1) % 500 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")
    

Epoch [500/10000], Loss: 0.1003
Epoch [1000/10000], Loss: 0.0973
Epoch [1500/10000], Loss: 0.0964
Epoch [2000/10000], Loss: 0.0959
Epoch [2500/10000], Loss: 0.0973
Epoch [3000/10000], Loss: 0.0954
Epoch [3500/10000], Loss: 0.0947
Epoch [4000/10000], Loss: 0.0950
Epoch [4500/10000], Loss: 0.0981
Epoch [5000/10000], Loss: 0.0941
Epoch [5500/10000], Loss: 0.0939
Epoch [6000/10000], Loss: 0.0941
Epoch [6500/10000], Loss: 0.0935
Epoch [7000/10000], Loss: 0.0944
Epoch [7500/10000], Loss: 0.0936
Epoch [8000/10000], Loss: 0.0930
Epoch [8500/10000], Loss: 0.0933
Epoch [9000/10000], Loss: 0.0930
Epoch [9500/10000], Loss: 0.0927
Epoch [10000/10000], Loss: 0.0926
